# Covalent KRAS co-folding — re-run of the arms lost on 22 Oct 2026

Re-creates the per-sample data for: Protenix-20250630 (all 21 entries, both arms), Boltz-2 (all 21 entries, both arms, plus the 40 pre-cutoff controls) and Protenix v1.0.0 on the 40 controls. Same runner (`kras_holdout.py`, sha256 d17a5e53…), same seed 42 / 5 samples / 200 steps, same scoring.

**Persistence rule:** every stage ends by writing `results.csv`, `pose_rmsd.csv`, the manifest and a tar of the predicted CIFs to **this account's** Drive at `MyDrive/kras_cofold_rerun_2026-10-23/`. The 22 Oct run wrote to a mount on an account that was over quota, and the FUSE listing showed the files while the upload silently failed. This time the folder is checked in the Drive web UI, not with `ls` on the mount.

Stages run from one `nohup` queue, so closing this tab does not stop them. Cell 5 shows progress.

## 1 · GPU and Drive

In [ ]:
import torch, os, datetime
print("cuda:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/kras_cofold_rerun_2026-10-23'
os.makedirs(DEST, exist_ok=True)
open(f"{DEST}/MOUNT_MARKER.txt","w").write(f"mounted {datetime.datetime.utcnow().isoformat()}Z\n")
print("DEST:", DEST, "->", os.listdir(DEST))
print("Now check that MOUNT_MARKER.txt shows up at drive.google.com before running the queue.")

## 2 · Write runner, scorers, manifests, queue

In [ ]:
import base64, pathlib, os
os.chdir('/content')
BLOBS = {
"kras_holdout.py": "...",
"protenix_arm.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJQcm90ZW5peCB0aGlyZCBhcm0gZm9yIHRoZSBjb3ZhbGVudCBLUkFTIGhvbGRvdXQuCgpSZWFkcyB3b3JrL21hbmlmZXN0Lmpzb24gd3JpdHRlbiBieSBga3Jhc19ob2xkb3V0LnB5IHByZXBhcmVgLCBzbyB0aGUgc2l0ZXMsCnNlcXVlbmNlcywgU01JTEVTIGFuZCB3YXJoZWFkIGF0b21zIGFyZSB0aGUgc2FtZSBvYmplY3RzIHRoZSBDaGFpLTEgYW5kIEJvbHR6LTIKYXJtcyB1c2VkLiBOb3RoaW5nIGlzIHJlLWRlcml2ZWQgYW5kIG5vdGhpbmcgY2FuIGRyaWZ0IGJldHdlZW4gYXJtcy4KClR3byBmYWN0cyBlc3RhYmxpc2hlZCBieSByZWFkaW5nIHByb3Rlbml4L2RhdGEvaW5mZXJlbmNlL2pzb25fcGFyc2VyLnB5IHJhdGhlcgp0aGFuIGluZmVycmluZyBmcm9tIGJlaGF2aW91cjoKCiAgMS4gUHJvdGVuaXggbmFtZXMgU01JTEVTIGxpZ2FuZCBhdG9tcyBlbGVtZW50IHN5bWJvbCArIHBlci1lbGVtZW50IGNvdW50ZXIgaW4KICAgICBSREtpdCBhdG9tIG9yZGVyIChyZGtpdF9tb2xfdG9fYXRvbV9pbmZvKS4gQ2hlbS5BZGRIcyBydW5zIGZpcnN0IGJ1dCBhcHBlbmRzCiAgICAgaHlkcm9nZW5zIGFmdGVyIHRoZSBoZWF2eSBhdG9tcywgc28gaGVhdnktYXRvbSBudW1iZXJpbmcgaXMgdW5jaGFuZ2VkLiBUaGlzCiAgICAgaXMgdGhlIFNBTUUgc2NoZW1lIGNoYWlfbGFiIDAuNi4xIHVzZXMgLS0gdmVyaWZpZWQgaWRlbnRpY2FsIG9uIGFsbCAyMQogICAgIGxpZ2FuZHMgaW4gdGhpcyBzZXQuIFNvIG1hbmlmZXN0Lmpzb24ncyBgbGF0b21fY2hhaWAgaXMgYWxzbyB0aGUgUHJvdGVuaXgKICAgICBuYW1lLCBhbmQgbm8gbmV3IG1hcHBpbmcgaXMgbmVlZGVkLgoKICAyLiBEZWNsYXJpbmcgdGhlIGJvbmQgbmVlZHMgbm8gQ0NEIGxvb2t1cCB3aGVuIHRoZSBsaWdhbmQgaXMgZ2l2ZW4gYXMgU01JTEVTOgogICAgIHRoZSBib25kIHJlZmVyZW5jZXMgZW50aXR5L3Bvc2l0aW9uL2F0b20gZGlyZWN0bHkuIFByb3Rlbml4IHRoZXJlZm9yZSBkb2VzCiAgICAgbm90IGluaGVyaXQgQm9sdHotMidzIGZyb3plbi1kaWN0aW9uYXJ5IGZhaWx1cmUsIHdoZXJlIDExIG9mIDEzIHBvc3QtY3V0b2ZmCiAgICAgbGlnYW5kcyBjb3VsZCBub3QgYmUgYm9uZC1kZWNsYXJlZCBhdCBhbGwuCgpFbnRpdHkgbnVtYmVyaW5nIGluIGBjb3ZhbGVudF9ib25kc2AgaXMgdGhlIDEtYmFzZWQgaW5kZXggaW50byBgc2VxdWVuY2VzYC4KCk1vZGVscyAoY29uZmlncy9jb25maWdzX21vZGVsX3R5cGUucHkpOgogIHByb3Rlbml4X2Jhc2VfZGVmYXVsdF92MS4wLjAgICAgdHJhaW5pbmcgY3V0b2ZmIDIwMjEtMDktMzAgIChBRjMtbWF0Y2hlZCkKICBwcm90ZW5peF9iYXNlXzIwMjUwNjMwX3YxLjAuMCAgIHRyYWluaW5nIGN1dG9mZiAyMDI1LTA2LTMwCgpVc2FnZToKICBweXRob24gcHJvdGVuaXhfYXJtLnB5IHByZXBhcmUgV09SS0RJUiBbLS10aWVyIEFdCiAgcHl0aG9uIHByb3Rlbml4X2FybS5weSBjbWRzICAgIFdPUktESVIgWy0tbW9kZWwgcHJvdGVuaXhfYmFzZV9kZWZhdWx0X3YxLjAuMF0KICBweXRob24gcHJvdGVuaXhfYXJtLnB5IGNvbGxlY3QgV09SS0RJUgoiIiIKaW1wb3J0IGFyZ3BhcnNlLCBqc29uLCBzaHV0aWwKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpOVUNMRU9USURFUyA9IHsiR0RQIiwgIkdOUCIsICJHVFAiLCAiR1NQIiwgIkdDUCJ9CklPTlMgPSB7Ik1HIn0KCgpkZWYgYnVpbGRfam9iKG0sIGFybSk6CiAgICAiIiJPbmUgUHJvdGVuaXggaW5mZXJlbmNlIGpvYiBkaWN0IGZvciBhbiBlbnRyeSBhbmQgYXJtLiIiIgogICAgc2VxcyA9IFt7InByb3RlaW5DaGFpbiI6IHsic2VxdWVuY2UiOiBtWyJzZXEiXSwgImNvdW50IjogMX19LAogICAgICAgICAgICB7ImxpZ2FuZCI6IHsibGlnYW5kIjogbVsic21pbGVzIl0sICJjb3VudCI6IDF9fV0KICAgIGZvciBjIGluIG1bImNvZmFjdG9ycyJdOgogICAgICAgIGlmIGMgaW4gSU9OUzoKICAgICAgICAgICAgc2Vxcy5hcHBlbmQoeyJpb24iOiB7ImlvbiI6IGMsICJjb3VudCI6IDF9fSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZXFzLmFwcGVuZCh7ImxpZ2FuZCI6IHsibGlnYW5kIjogZiJDQ0Rfe2N9IiwgImNvdW50IjogMX19KQoKICAgIGpvYiA9IHsibmFtZSI6IGYie21bJ2VudHJ5J119X3thcm19IiwgInNlcXVlbmNlcyI6IHNlcXN9CgogICAgaWYgYXJtID09ICJib25kIjoKICAgICAgICBqb2JbImNvdmFsZW50X2JvbmRzIl0gPSBbewogICAgICAgICAgICAiZW50aXR5MSI6ICIxIiwgInBvc2l0aW9uMSI6IHN0cihtWyJzZXFfaW5kZXgiXSksICJhdG9tMSI6IG1bInBhdG9tIl0sCiAgICAgICAgICAgICJlbnRpdHkyIjogIjIiLCAicG9zaXRpb24yIjogIjEiLCAgICAgICAgICAgICAgICAgImF0b20yIjogbVsibGF0b21fY2hhaSJdLAogICAgICAgIH1dCiAgICByZXR1cm4gam9iCgoKZGVmIHByZXBhcmUod29yaywgdGllcik6CiAgICB3b3JrID0gUGF0aCh3b3JrKQogICAgbWFuID0ganNvbi5sb2Fkcygod29yayAvICJtYW5pZmVzdC5qc29uIikucmVhZF90ZXh0KCkpCgogICAgIyBrcmFzX2hvbGRvdXQucHJlcGFyZSBkb2VzIG5vdCBzdG9yZSB0aGUgc2VxdWVuY2U7IHJlY292ZXIgaXQgdGhlIHNhbWUgd2F5LgogICAgaW1wb3J0IHN5cwogICAgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50KSkKICAgIGZyb20ga3Jhc19ob2xkb3V0IGltcG9ydCBlbnRpdHlfc2VxCgogICAgb3V0X2RpciA9IHdvcmsgLyAiaW5wdXRzIiAvICJwcm90ZW5peCIKICAgIG91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgbWFkZSwgc2tpcHBlZCA9IFtdLCBbXQoKICAgIGZvciBtIGluIG1hbjoKICAgICAgICBpZiB0aWVyIGFuZCBub3QgbVsidGllciJdLnN0YXJ0c3dpdGgodGllcik6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgY2lmID0gd29yayAvICJzdHJ1Y3R1cmVzIiAvIGYie21bJ2VudHJ5J119LmNpZiIKICAgICAgICBzZXEsIF8gPSBlbnRpdHlfc2VxKGNpZiwgbVsiY2hhaW4iXSkKICAgICAgICBtID0gZGljdChtLCBzZXE9c2VxKQogICAgICAgIGZvciBhcm0gaW4gKCJzbWlsZXMiLCAiYm9uZCIpOgogICAgICAgICAgICBpZiBhcm0gPT0gImJvbmQiIGFuZCBub3QgbS5nZXQoImxhdG9tX2NoYWkiKToKICAgICAgICAgICAgICAgIHNraXBwZWQuYXBwZW5kKGYie21bJ2VudHJ5J119OiB3YXJoZWFkIGFtYmlndW91cyBpbiBTTUlMRVMgb3JkZXIsIGJvbmQgYXJtIHNraXBwZWQiKQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcCA9IG91dF9kaXIgLyBmInttWydlbnRyeSddfV97YXJtfS5qc29uIgogICAgICAgICAgICBwLndyaXRlX3RleHQoanNvbi5kdW1wcyhbYnVpbGRfam9iKG0sIGFybSldLCBpbmRlbnQ9MikpCiAgICAgICAgICAgIG1hZGUuYXBwZW5kKHAubmFtZSkKICAgICAgICBwcmludChmInttWydlbnRyeSddfSAge21bJ3JlcyddfXttWydzZXFfaW5kZXgnXX0ue21bJ3BhdG9tJ119IC0+ICIKICAgICAgICAgICAgICBmInttWydsaWcnXX0ue21bJ2xhdG9tX2NoYWknXX0gIGNvZj17JywnLmpvaW4obVsnY29mYWN0b3JzJ10pIG9yICctJ30gIHNlcT17bGVuKHNlcSl9IikKCiAgICBwcmludChmIlxud3JvdGUge2xlbihtYWRlKX0gam9iIGZpbGVzIHRvIHtvdXRfZGlyfSIpCiAgICBmb3IgcyBpbiBza2lwcGVkOgogICAgICAgIHByaW50KCIgIFNLSVAiLCBzKQogICAgcmV0dXJuIG1hZGUKCgpkZWYgY21kcyh3b3JrLCBtb2RlbCk6CiAgICB3b3JrID0gUGF0aCh3b3JrKQogICAgZCA9IHdvcmsgLyAiaW5wdXRzIiAvICJwcm90ZW5peCIKICAgIGxpbmVzID0gW10KICAgIGZvciBwIGluIHNvcnRlZChkLmdsb2IoIiouanNvbiIpKToKICAgICAgICBvdXQgPSB3b3JrIC8gIm91dF9yYXciIC8gInByb3Rlbml4IiAvIHAuc3RlbQogICAgICAgIGxpbmVzLmFwcGVuZCgKICAgICAgICAgICAgZidwcm90ZW5peCBwcmVkIC1pICJ7cH0iIC1vICJ7b3V0fSIgLXMgNDIgLW4ge21vZGVsfSAnCiAgICAgICAgICAgIGYnLWUgNSAtcCAyMDAgLWMgMTAgLS11c2VfZGVmYXVsdF9wYXJhbXMgdHJ1ZScKICAgICAgICApCiAgICBwcmludCgiXG4iLmpvaW4obGluZXMpKQogICAgcmV0dXJuIGxpbmVzCgoKZGVmIGNvbGxlY3Qod29yayk6CiAgICAiIiJGbGF0dGVuIFByb3Rlbml4IG91dHB1dCBpbnRvIHdvcmsvb3V0L3Byb3Rlbml4LzxlbnRyeT5fPGFybT4vIGZvciBzY29yaW5nLiIiIgogICAgd29yayA9IFBhdGgod29yaykKICAgIHJhdyA9IHdvcmsgLyAib3V0X3JhdyIgLyAicHJvdGVuaXgiCiAgICBpZiBub3QgcmF3LmV4aXN0cygpOgogICAgICAgIHByaW50KCJubyBvdXRwdXQgYXQiLCByYXcpOyByZXR1cm4KICAgIG4gPSAwCiAgICBmb3Igam9iIGluIHNvcnRlZChyYXcuaXRlcmRpcigpKToKICAgICAgICBpZiBub3Qgam9iLmlzX2RpcigpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGRlc3QgPSB3b3JrIC8gIm91dCIgLyAicHJvdGVuaXgiIC8gam9iLm5hbWUKICAgICAgICBkZXN0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBjaWZzID0gc29ydGVkKGpvYi5yZ2xvYigiKi5jaWYiKSkKICAgICAgICBmb3IgaSwgYyBpbiBlbnVtZXJhdGUoY2lmcyk6CiAgICAgICAgICAgIHNodXRpbC5jb3B5KGMsIGRlc3QgLyBmInNhbXBsZV97aX0uY2lmIikKICAgICAgICAgICAgbiArPSAxCiAgICAgICAgcHJpbnQoZiJ7am9iLm5hbWV9OiB7bGVuKGNpZnMpfSBtb2RlbHMiKQogICAgcHJpbnQoZiJcbmNvbGxlY3RlZCB7bn0gc3RydWN0dXJlcyBpbnRvIHt3b3JrLydvdXQnLydwcm90ZW5peCd9IikKICAgIHByaW50KCJub3cgcnVuOiAgcHl0aG9uIGtyYXNfaG9sZG91dC5weSBzY29yZSIsIHdvcmspCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCJjbWQiLCBjaG9pY2VzPVsicHJlcGFyZSIsICJjbWRzIiwgImNvbGxlY3QiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgid29yayIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdGllciIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tb2RlbCIsIGRlZmF1bHQ9InByb3Rlbml4X2Jhc2VfZGVmYXVsdF92MS4wLjAiKQogICAgYSA9IGFwLnBhcnNlX2FyZ3MoKQogICAgaWYgYS5jbWQgPT0gInByZXBhcmUiOgogICAgICAgIHByZXBhcmUoYS53b3JrLCBhLnRpZXIpCiAgICBlbGlmIGEuY21kID09ICJjbWRzIjoKICAgICAgICBjbWRzKGEud29yaywgYS5tb2RlbCkKICAgIGVsc2U6CiAgICAgICAgY29sbGVjdChhLndvcmspCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
"poserms.py": "IiIiTGlnYW5kIHBvc2UgUk1TRCB2cyBkZXBvc2l0ZWQsIHN5bW1ldHJ5LWNvcnJlY3RlZCwgZm9yIGNvLWZvbGRpbmcgcHJlZGljdGlvbnMuCgpTdXBlcnBvc2VzIHRoZSBwcmVkaWN0ZWQgcHJvdGVpbiBvbnRvIHRoZSBkZXBvc2l0ZWQgcHJvdGVpbiB1c2luZyBDQSBhdG9tcyBvZiB0aGUKcmVzaWR1ZXMgcHJlc2VudCBpbiBib3RoLCBhcHBsaWVzIHRoYXQgdHJhbnNmb3JtIHRvIHRoZSBwcmVkaWN0ZWQgbGlnYW5kLCB0aGVuCnJlcG9ydHMgdGhlIG1pbmltdW0gaGVhdnktYXRvbSBSTVNEIG92ZXIgdGhlIGxpZ2FuZCdzIGF1dG9tb3JwaGlzbSBncm91cC4KCkF0b20gY29ycmVzcG9uZGVuY2U6CiAgcHJlZGljdGVkIGxpZ2FuZCBhdG9tcyBhcmUgbmFtZWQgZWxlbWVudCtwZXItZWxlbWVudC1jb3VudGVyIGluIFJES2l0IG9yZGVyIG9mCiAgdGhlIFNNSUxFUyAodmVyaWZpZWQgaWRlbnRpY2FsIGZvciBjaGFpX2xhYiAwLjYuMSBhbmQgUHJvdGVuaXggdjEuMC4wKSwgc28gdGhlCiAgcHJlZGljdGVkIG5hbWUgbWFwcyBkaXJlY3RseSBvbnRvIGEgU01JTEVTIG1vbCBhdG9tIGluZGV4LgogIGRlcG9zaXRlZCBsaWdhbmQgYXRvbXMgY2FycnkgQ0NEIG5hbWVzLCBtYXBwZWQgb250byBTTUlMRVMgaW5kaWNlcyBieSBhIGdyYXBoCiAgbWF0Y2ggYmV0d2VlbiB0aGUgQ0NEIG1vbCBhbmQgdGhlIFNNSUxFUyBtb2wuCgpFbWl0cyBvbmUgcm93IHBlciAobW9kZWwsIGVudHJ5LCBhcm0sIHNhbXBsZSkgd2l0aCBwb3NlIFJNU0QgYWxvbmdzaWRlIHRoZQpudWNsZW9waGlsZS10by1uZWFyZXN0LWxpZ2FuZC1oZWF2eS1hdG9tIGRpc3RhbmNlLCBzbyB0aGUgdHdvIG1ldHJpY3MgY2FuIGJlCmNvbXBhcmVkIGRpcmVjdGx5LgoiIiIKaW1wb3J0IGNzdiwganNvbiwgbWF0aCwgc3lzCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBnZW1taQpmcm9tIHJka2l0IGltcG9ydCBDaGVtLCBSRExvZ2dlcgpSRExvZ2dlci5EaXNhYmxlTG9nKCJyZEFwcC4qIikKCnN5cy5wYXRoLmluc2VydCgwLCAiL2NvbnRlbnQiKQpmcm9tIGtyYXNfaG9sZG91dCBpbXBvcnQgY2NkX21vbCwgY2NkX3NtaWxlcywgY2hhaV9uYW1lcywgc2tlbGV0b24sIHNtaWxlc19tb2wsIGVudGl0eV9zZXEKCgpkZWYgYm9sdHpfbmFtZXMoc21pKToKICAgICIiIkJvbHR6LTIgbmFtZXMgU01JTEVTIGxpZ2FuZCBhdG9tcyBlbGVtZW50ICsgY2Fub25pY2FsIHJhbmsgb3ZlciB0aGUgSC1hZGRlZCBtb2wKICAgIChib2x0ei9kYXRhL3BhcnNlL3NjaGVtYS5weSkuIEhlYXZ5IGF0b21zIGtlZXAgdGhlaXIgaGVhdnktYXRvbSBpbmRpY2VzIGJlY2F1c2UgQWRkSHMgYXBwZW5kcy4iIiIKICAgIGZyb20gcmRraXQuQ2hlbSBpbXBvcnQgQWxsQ2hlbQogICAgbSA9IEFsbENoZW0uQWRkSHMoQWxsQ2hlbS5Nb2xGcm9tU21pbGVzKHNtaSkpCiAgICByYW5rID0gbGlzdChBbGxDaGVtLkNhbm9uaWNhbFJhbmtBdG9tcyhtKSkKICAgIG5oZWF2eSA9IHNtaWxlc19tb2woc21pKS5HZXROdW1BdG9tcygpCiAgICByZXR1cm4gWyhtLkdldEF0b21XaXRoSWR4KGkpLkdldFN5bWJvbCgpLnVwcGVyKCkgKyBzdHIocmFua1tpXSArIDEpKSBmb3IgaSBpbiByYW5nZShuaGVhdnkpXQoKQUEzID0geyJBTEEiLCJBUkciLCJBU04iLCJBU1AiLCJDWVMiLCJHTE4iLCJHTFUiLCJHTFkiLCJISVMiLCJJTEUiLCJMRVUiLCJMWVMiLAogICAgICAgIk1FVCIsIlBIRSIsIlBSTyIsIlNFUiIsIlRIUiIsIlRSUCIsIlRZUiIsIlZBTCIsIk1TRSIsIlNFQyIsIlBZTCJ9CkNPRiA9IHsiSE9IIiwiR0RQIiwiR05QIiwiR1RQIiwiR1NQIiwiR0NQIiwiTUcifQoKCmRlZiBrYWJzY2goUCwgUSk6CiAgICAiIiJSb3RhdGlvbit0cmFuc2xhdGlvbiB0YWtpbmcgUCBvbnRvIFEgKGJvdGggKE4sMykpOyByZXR1cm5zIFIsIHQuIiIiCiAgICBwYywgcWMgPSBQLm1lYW4oMCksIFEubWVhbigwKQogICAgSCA9IChQIC0gcGMpLlQgQCAoUSAtIHFjKQogICAgVSwgUywgVnQgPSBucC5saW5hbGcuc3ZkKEgpCiAgICBkID0gbnAuc2lnbihucC5saW5hbGcuZGV0KFZ0LlQgQCBVLlQpKQogICAgRCA9IG5wLmRpYWcoWzEuMCwgMS4wLCBkXSkKICAgIFIgPSBWdC5UIEAgRCBAIFUuVAogICAgcmV0dXJuIFIsIHFjIC0gUiBAIHBjCgoKZGVmIGNjZF90b19zbWlsZXNfbWFwKHJlZiwgc21pKToKICAgICIiIkZ1bGwgbWFwIHtDQ0QgYXRvbSBuYW1lIC0+IFNNSUxFUyBtb2wgYXRvbSBpbmRleH0sIG9yIE5vbmUgaWYgYW1iaWd1b3VzLiIiIgogICAgcHJvYmUgPSBzbWlsZXNfbW9sKHNtaSkKICAgIGlmIHByb2JlIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIG1hdGNoZXMgPSBza2VsZXRvbihyZWYpLkdldFN1YnN0cnVjdE1hdGNoZXMoc2tlbGV0b24ocHJvYmUpLCB1bmlxdWlmeT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdXNlQ2hpcmFsaXR5PUZhbHNlLCBtYXhNYXRjaGVzPTIwMDAwKQogICAgaWYgbm90IG1hdGNoZXM6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIG5hbWVzID0gW2EuR2V0UHJvcCgibmFtZSIpIGZvciBhIGluIHJlZi5HZXRBdG9tcygpXQogICAgIyBtYXRjaGVzW2tdW3NtaWxlc19pZHhdID0gcmVmX2lkeCA7IGludmVydCwga2VlcCBvbmx5IHVuYW1iaWd1b3VzIGFzc2lnbm1lbnRzCiAgICBwZXJfcmVmID0gZGVmYXVsdGRpY3Qoc2V0KQogICAgZm9yIG0gaW4gbWF0Y2hlczoKICAgICAgICBmb3Igc21pX2lkeCwgcmVmX2lkeCBpbiBlbnVtZXJhdGUobSk6CiAgICAgICAgICAgIHBlcl9yZWZbcmVmX2lkeF0uYWRkKHNtaV9pZHgpCiAgICBvdXQgPSB7fQogICAgZm9yIHJlZl9pZHgsIGNhbmRzIGluIHBlcl9yZWYuaXRlbXMoKToKICAgICAgICBpZiBsZW4oY2FuZHMpID09IDE6CiAgICAgICAgICAgIG91dFtuYW1lc1tyZWZfaWR4XV0gPSBuZXh0KGl0ZXIoY2FuZHMpKQogICAgcmV0dXJuIG91dCBvciBOb25lCgoKZGVmIGF1dG9tb3JwaGlzbXMobW9sLCBjYXA9NTAwMCk6CiAgICBtcyA9IG1vbC5HZXRTdWJzdHJ1Y3RNYXRjaGVzKG1vbCwgdW5pcXVpZnk9RmFsc2UsIHVzZUNoaXJhbGl0eT1GYWxzZSwgbWF4TWF0Y2hlcz1jYXApCiAgICByZXR1cm4gbXMgaWYgbXMgZWxzZSBbdHVwbGUocmFuZ2UobW9sLkdldE51bUF0b21zKCkpKV0KCgpkZWYgcHJvdGVpbl9jYShzdCwgd2FudF9jaGFpbj1Ob25lKToKICAgIG91dCA9IHt9CiAgICBmb3IgY2ggaW4gc3RbMF06CiAgICAgICAgZm9yIHJlcyBpbiBjaDoKICAgICAgICAgICAgaWYgcmVzLm5hbWUgaW4gQUEzOgogICAgICAgICAgICAgICAgZm9yIGEgaW4gcmVzOgogICAgICAgICAgICAgICAgICAgIGlmIGEubmFtZSA9PSAiQ0EiOgogICAgICAgICAgICAgICAgICAgICAgICBvdXRbcmVzLnNlcWlkLm51bV0gPSBucC5hcnJheShbYS5wb3MueCwgYS5wb3MueSwgYS5wb3Muel0pCiAgICByZXR1cm4gb3V0CgoKZGVmIGxpZ2FuZF9hdG9tcyhzdCwgZXhjbHVkZT1DT0YpOgogICAgb3V0ID0ge30KICAgIGZvciBjaCBpbiBzdFswXToKICAgICAgICBmb3IgcmVzIGluIGNoOgogICAgICAgICAgICBpZiByZXMubmFtZSBpbiBBQTMgb3IgcmVzLm5hbWUgaW4gZXhjbHVkZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBhIGluIHJlczoKICAgICAgICAgICAgICAgIGlmIGEuZWxlbWVudC5uYW1lICE9ICJIIjoKICAgICAgICAgICAgICAgICAgICBvdXRbYS5uYW1lXSA9IG5wLmFycmF5KFthLnBvcy54LCBhLnBvcy55LCBhLnBvcy56XSkKICAgIHJldHVybiBvdXQKCgpkZWYgbnVjbGVvcGhpbGUoc3QsIHNlcV9pbmRleCwgcmVzbnVtLCBwYXRvbSk6CiAgICBmb3IgY2ggaW4gc3RbMF06CiAgICAgICAgZm9yIHJlcyBpbiBjaDoKICAgICAgICAgICAgaWYgcmVzLm5hbWUgaW4gQUEzIGFuZCByZXMuc2VxaWQubnVtIGluIChzZXFfaW5kZXgsIHJlc251bSk6CiAgICAgICAgICAgICAgICBmb3IgYSBpbiByZXM6CiAgICAgICAgICAgICAgICAgICAgaWYgYS5uYW1lID09IHBhdG9tOgogICAgICAgICAgICAgICAgICAgICAgICByZXR1cm4gbnAuYXJyYXkoW2EucG9zLngsIGEucG9zLnksIGEucG9zLnpdKQogICAgcmV0dXJuIE5vbmUKCgpkZWYgbWFpbih3b3JrZGlyKToKICAgIFcgPSBQYXRoKHdvcmtkaXIpCiAgICBtYW4gPSB7bVsiZW50cnkiXTogbSBmb3IgbSBpbiBqc29uLmxvYWRzKChXIC8gIm1hbmlmZXN0Lmpzb24iKS5yZWFkX3RleHQoKSl9CiAgICByb3dzLCBwcm9ibGVtcyA9IFtdLCBbXQoKICAgIGZvciBtb2RlbCBpbiAoImNoYWkiLCAiYm9sdHoiLCAicHJvdGVuaXgiKToKICAgICAgICByb290ID0gVyAvICJvdXQiIC8gbW9kZWwKICAgICAgICBpZiBub3Qgcm9vdC5leGlzdHMoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3Igam9iZGlyIGluIHNvcnRlZChwIGZvciBwIGluIHJvb3QuaXRlcmRpcigpIGlmIHAuaXNfZGlyKCkpOgogICAgICAgICAgICBzdGVtID0gam9iZGlyLm5hbWUKICAgICAgICAgICAgZW50cnkgPSBzdGVtLnNwbGl0KCJfIilbMF0KICAgICAgICAgICAgaWYgZW50cnkgbm90IGluIG1hbjoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGFybSA9ICJib25kIiBpZiBzdGVtLmVuZHN3aXRoKCJfYm9uZCIpIGVsc2UgInNtaWxlcyIKICAgICAgICAgICAgbSA9IG1hbltlbnRyeV0KCiAgICAgICAgICAgIGRlcF9wYXRoID0gVyAvICJzdHJ1Y3R1cmVzIiAvIGYie2VudHJ5fS5jaWYiCiAgICAgICAgICAgIGxpZ19jaWYgPSBXIC8gInN0cnVjdHVyZXMiIC8gZiJ7bVsnbGlnJ119LmNpZiIKICAgICAgICAgICAgaWYgbm90IGRlcF9wYXRoLmV4aXN0cygpIG9yIG5vdCBsaWdfY2lmLmV4aXN0cygpOgogICAgICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKGYie3N0ZW19OiBtaXNzaW5nIGRlcG9zaXRlZCBmaWxlcyIpOyBjb250aW51ZQoKICAgICAgICAgICAgZGVwID0gZ2VtbWkucmVhZF9zdHJ1Y3R1cmUoc3RyKGRlcF9wYXRoKSkKICAgICAgICAgICAgZGVwLnNldHVwX2VudGl0aWVzKCkKICAgICAgICAgICAgZGVwLnJlbW92ZV9hbHRlcm5hdGl2ZV9jb25mb3JtYXRpb25zKCkKICAgICAgICAgICAgc21pID0gbVsic21pbGVzIl0KICAgICAgICAgICAgbW9sID0gc21pbGVzX21vbChzbWkpCiAgICAgICAgICAgIGlmIG1vbCBpcyBOb25lOgogICAgICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKGYie3N0ZW19OiBTTUlMRVMgdW5wYXJzZWFibGUiKTsgY29udGludWUKICAgICAgICAgICAgbmFtZXMgPSBib2x0el9uYW1lcyhzbWkpIGlmIG1vZGVsID09ICJib2x0eiIgZWxzZSBjaGFpX25hbWVzKHNtaSkKICAgICAgICAgICAgbm1hcCA9IHtuOiBpIGZvciBpLCBuIGluIGVudW1lcmF0ZShuYW1lcyl9CiAgICAgICAgICAgIGNtYXAgPSBjY2RfdG9fc21pbGVzX21hcChjY2RfbW9sKGxpZ19jaWYpLCBzbWkpCiAgICAgICAgICAgIGlmIG5vdCBjbWFwOgogICAgICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKGYie3N0ZW19OiBubyBDQ0QtPlNNSUxFUyBtYXAiKTsgY29udGludWUKCiAgICAgICAgICAgIGRlcF9saWdfbmFtZWQgPSBsaWdhbmRfYXRvbXMoZGVwKQogICAgICAgICAgICBkZXBfY29vcmRzID0ge30KICAgICAgICAgICAgZm9yIG5tLCBwb3MgaW4gZGVwX2xpZ19uYW1lZC5pdGVtcygpOgogICAgICAgICAgICAgICAgaWYgbm0gaW4gY21hcDoKICAgICAgICAgICAgICAgICAgICBkZXBfY29vcmRzW2NtYXBbbm1dXSA9IHBvcwogICAgICAgICAgICBpZiBsZW4oZGVwX2Nvb3JkcykgPCA0OgogICAgICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKGYie3N0ZW19OiBkZXBvc2l0ZWQgbGlnYW5kIGF0b21zIHVubWFwcGVkIik7IGNvbnRpbnVlCgogICAgICAgICAgICBfLCBhbWFwID0gZW50aXR5X3NlcShkZXBfcGF0aCwgbVsiY2hhaW4iXSkKICAgICAgICAgICAgZGVwX2NhX2F1dGggPSBwcm90ZWluX2NhKGRlcCkKICAgICAgICAgICAgZGVwX2NhX2lkeCA9IHthbWFwW2FdOiB4eXogZm9yIGEsIHh5eiBpbiBkZXBfY2FfYXV0aC5pdGVtcygpIGlmIGEgaW4gYW1hcH0KCiAgICAgICAgICAgIGF1dG9zID0gYXV0b21vcnBoaXNtcyhtb2wpCgogICAgICAgICAgICBmb3IgY2lmIGluIHNvcnRlZChqb2JkaXIuZ2xvYigiKi5jaWYiKSk6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcHJlZCA9IGdlbW1pLnJlYWRfc3RydWN0dXJlKHN0cihjaWYpKQogICAgICAgICAgICAgICAgICAgIHByZWQuc2V0dXBfZW50aXRpZXMoKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleDoKICAgICAgICAgICAgICAgICAgICBwcm9ibGVtcy5hcHBlbmQoZiJ7Y2lmLm5hbWV9OiB7ZXh9Iik7IGNvbnRpbnVlCgogICAgICAgICAgICAgICAgcHJlZF9jYSA9IHByb3RlaW5fY2EocHJlZCkKICAgICAgICAgICAgICAgIGNvbW1vbiA9IHNvcnRlZChzZXQocHJlZF9jYSkgJiBzZXQoZGVwX2NhX2lkeCkpCiAgICAgICAgICAgICAgICBpZiBsZW4oY29tbW9uKSA8IDIwOgogICAgICAgICAgICAgICAgICAgIHByb2JsZW1zLmFwcGVuZChmIntzdGVtfS97Y2lmLm5hbWV9OiBvbmx5IHtsZW4oY29tbW9uKX0gY29tbW9uIENBIik7IGNvbnRpbnVlCiAgICAgICAgICAgICAgICBQID0gbnAuYXJyYXkoW3ByZWRfY2FbaV0gZm9yIGkgaW4gY29tbW9uXSkKICAgICAgICAgICAgICAgIFEgPSBucC5hcnJheShbZGVwX2NhX2lkeFtpXSBmb3IgaSBpbiBjb21tb25dKQogICAgICAgICAgICAgICAgUiwgdCA9IGthYnNjaChQLCBRKQogICAgICAgICAgICAgICAgY2Ffcm1zZCA9IGZsb2F0KG5wLnNxcnQoKCgoUCBAIFIuVCkgKyB0IC0gUSkgKiogMikuc3VtKDEpLm1lYW4oKSkpCgogICAgICAgICAgICAgICAgcHJlZF9saWcgPSBsaWdhbmRfYXRvbXMocHJlZCkKICAgICAgICAgICAgICAgIHByZWRfY29vcmRzID0ge30KICAgICAgICAgICAgICAgIGZvciBubSwgcG9zIGluIHByZWRfbGlnLml0ZW1zKCk6CiAgICAgICAgICAgICAgICAgICAgaWYgbm0gaW4gbm1hcDoKICAgICAgICAgICAgICAgICAgICAgICAgcHJlZF9jb29yZHNbbm1hcFtubV1dID0gKFIgQCBwb3MpICsgdAogICAgICAgICAgICAgICAgc2hhcmVkID0gc29ydGVkKHNldChwcmVkX2Nvb3JkcykgJiBzZXQoZGVwX2Nvb3JkcykpCiAgICAgICAgICAgICAgICBpZiBsZW4oc2hhcmVkKSA8IDQ6CiAgICAgICAgICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKGYie3N0ZW19L3tjaWYubmFtZX06IHtsZW4oc2hhcmVkKX0gc2hhcmVkIGxpZ2FuZCBhdG9tcyIpOyBjb250aW51ZQoKICAgICAgICAgICAgICAgIGJlc3QgPSBOb25lCiAgICAgICAgICAgICAgICBmb3IgYXV0byBpbiBhdXRvczoKICAgICAgICAgICAgICAgICAgICAjIGF1dG8gbWFwcyBjYW5vbmljYWwgaW5kZXggLT4gcGVybXV0ZWQgaW5kZXgKICAgICAgICAgICAgICAgICAgICBwdHMsIHJlZiA9IFtdLCBbXQogICAgICAgICAgICAgICAgICAgIG9rID0gVHJ1ZQogICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHNoYXJlZDoKICAgICAgICAgICAgICAgICAgICAgICAgaiA9IGF1dG9baV0KICAgICAgICAgICAgICAgICAgICAgICAgaWYgaiBub3QgaW4gcHJlZF9jb29yZHM6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBvayA9IEZhbHNlOyBicmVhawogICAgICAgICAgICAgICAgICAgICAgICBwdHMuYXBwZW5kKHByZWRfY29vcmRzW2pdKTsgcmVmLmFwcGVuZChkZXBfY29vcmRzW2ldKQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCBvayBvciBub3QgcHRzOgogICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIGQgPSBucC5zcXJ0KCgoKG5wLmFycmF5KHB0cykgLSBucC5hcnJheShyZWYpKSAqKiAyKS5zdW0oMSkpLm1lYW4oKSkKICAgICAgICAgICAgICAgICAgICBpZiBiZXN0IGlzIE5vbmUgb3IgZCA8IGJlc3Q6CiAgICAgICAgICAgICAgICAgICAgICAgIGJlc3QgPSBmbG9hdChkKQogICAgICAgICAgICAgICAgaWYgYmVzdCBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHByb2JsZW1zLmFwcGVuZChmIntzdGVtfS97Y2lmLm5hbWV9OiBubyB2YWxpZCBhdXRvbW9ycGhpc20iKTsgY29udGludWUKCiAgICAgICAgICAgICAgICBudWMgPSBudWNsZW9waGlsZShwcmVkLCBtWyJzZXFfaW5kZXgiXSwgbVsicmVzbnVtIl0sIG1bInBhdG9tIl0pCiAgICAgICAgICAgICAgICBuZWFyID0gTm9uZQogICAgICAgICAgICAgICAgaWYgbnVjIGlzIG5vdCBOb25lIGFuZCBwcmVkX2xpZzoKICAgICAgICAgICAgICAgICAgICBuZWFyID0gbWluKGZsb2F0KG5wLmxpbmFsZy5ub3JtKG51YyAtIHApKSBmb3IgcCBpbiBwcmVkX2xpZy52YWx1ZXMoKSkKCiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZChkaWN0KG1vZGVsPW1vZGVsLCBlbnRyeT1lbnRyeSwgdGllcj1tWyJ0aWVyIl0sIGFybT1hcm0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNhbXBsZT1jaWYubmFtZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcG9zZV9ybXNkX0E9cm91bmQoYmVzdCwgMyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNhX3Jtc2RfQT1yb3VuZChjYV9ybXNkLCAzKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGlzdF9uZWFyZXN0X2xpZ19hdG9tX0E9cm91bmQobmVhciwgMykgaWYgbmVhciBpcyBub3QgTm9uZSBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5fbGlnX2F0b21zX21hdGNoZWQ9bGVuKHNoYXJlZCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRlcG9zaXRlZF9ib25kX0E9bVsiZGVwb3NpdGVkX2JvbmRfQSJdKSkKCiAgICBpZiBub3Qgcm93czoKICAgICAgICBwcmludCgibm8gcm93cyBwcm9kdWNlZCIpOyByZXR1cm4KICAgIG91dHAgPSBXIC8gInBvc2Vfcm1zZC5jc3YiCiAgICB3aXRoIG9wZW4ob3V0cCwgInciLCBuZXdsaW5lPSIiKSBhcyBmaDoKICAgICAgICB3ID0gY3N2LkRpY3RXcml0ZXIoZmgsIGZpZWxkbmFtZXM9bGlzdChyb3dzWzBdKSk7IHcud3JpdGVoZWFkZXIoKTsgdy53cml0ZXJvd3Mocm93cykKCiAgICBhZ2cgPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIHIgaW4gcm93czoKICAgICAgICBhZ2dbKHJbIm1vZGVsIl0sIHJbInRpZXIiXSwgclsiYXJtIl0pXS5hcHBlbmQocikKICAgIHByaW50KGYieydtb2RlbCc6PDl9eyd0aWVyJzo8MjJ9eydhcm0nOjw4fXsnbic6PjR9eydwb3NlUk1TRCc6PjEwfXsnPD0yQSBwb3NlJzo+MTF9eydib25kPD0yQSc6PjEwfSIpCiAgICBmb3IgayBpbiBzb3J0ZWQoYWdnKToKICAgICAgICB2ID0gYWdnW2tdCiAgICAgICAgcHIgPSBzb3J0ZWQoeFsicG9zZV9ybXNkX0EiXSBmb3IgeCBpbiB2KQogICAgICAgIG1lZCA9IHByW2xlbihwcikvLzJdIGlmIGxlbihwcikgJSAyIGVsc2UgKHByW2xlbihwcikvLzItMV0rcHJbbGVuKHByKS8vMl0pLzIKICAgICAgICBucG9zZSA9IHN1bSgxIGZvciB4IGluIHYgaWYgeFsicG9zZV9ybXNkX0EiXSA8PSAyLjApCiAgICAgICAgbmJvbmQgPSBzdW0oMSBmb3IgeCBpbiB2IGlmIHhbImRpc3RfbmVhcmVzdF9saWdfYXRvbV9BIl0gaXMgbm90IE5vbmUKICAgICAgICAgICAgICAgICAgICBhbmQgeFsiZGlzdF9uZWFyZXN0X2xpZ19hdG9tX0EiXSA8PSAyLjApCiAgICAgICAgcHJpbnQoZiJ7a1swXTo8OX17a1sxXTo8MjJ9e2tbMl06PDh9e2xlbih2KTo+NH17bWVkOj4xMC4yZn0iCiAgICAgICAgICAgICAgZiJ7c3RyKG5wb3NlKSsnLycrc3RyKGxlbih2KSk6PjExfXtzdHIobmJvbmQpKycvJytzdHIobGVuKHYpKTo+MTB9IikKCiAgICBkaXNhZ3JlZSA9IFtyIGZvciByIGluIHJvd3MgaWYgclsiZGlzdF9uZWFyZXN0X2xpZ19hdG9tX0EiXSBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgYW5kIHJbInBvc2Vfcm1zZF9BIl0gPD0gMi4wIGFuZCByWyJkaXN0X25lYXJlc3RfbGlnX2F0b21fQSJdID4gMi4wXQogICAgcHJpbnQoZiJcbnNhbXBsZXMgYSBwb3NlIGN1dCBhY2NlcHRzIGJ1dCB0aGUgYm9uZCBtZXRyaWMgcmVqZWN0czoge2xlbihkaXNhZ3JlZSl9IC8ge2xlbihyb3dzKX0iKQogICAgZm9yIHIgaW4gc29ydGVkKGRpc2FncmVlLCBrZXk9bGFtYmRhIHg6IHhbInBvc2Vfcm1zZF9BIl0pWzoxNV06CiAgICAgICAgcHJpbnQoZiIgIHtyWydtb2RlbCddOjw5fXtyWydlbnRyeSddfSB7clsnYXJtJ106PDd9IHBvc2Uge3JbJ3Bvc2Vfcm1zZF9BJ106PjUuMmZ9ICAiCiAgICAgICAgICAgICAgZiJuZWFyZXN0IHtyWydkaXN0X25lYXJlc3RfbGlnX2F0b21fQSddOj42LjJmfSIpCiAgICBwcmludChmIlxud3JvdGUge291dHB9ICAoe2xlbihyb3dzKX0gcm93cykiKQogICAgaWYgcHJvYmxlbXM6CiAgICAgICAgcHJpbnQoZiJcbntsZW4ocHJvYmxlbXMpfSBwcm9ibGVtczoiKQogICAgICAgIGZvciBwIGluIHByb2JsZW1zWzoxNV06CiAgICAgICAgICAgIHByaW50KCIgICIsIHApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oc3lzLmFyZ3ZbMV0gaWYgbGVuKHN5cy5hcmd2KSA+IDEgZWxzZSAid29ya19weCIpCg==",
"collect.py": "IiIiRmxhdHRlbiByYXcgbW9kZWwgb3V0cHV0IGludG8gb3V0Lzxtb2RlbD4vPGVudHJ5Pl88YXJtPi9zYW1wbGVfaS5jaWYuCgpDb3BpZXMgZnJvbSBvdXRfcmF3LyAob3V0c2lkZSBvdXQvKSBzbyBhIHJlY3Vyc2l2ZSBnbG9iIG92ZXIgb3V0LyBzZWVzIGVhY2ggc3RydWN0dXJlIG9uY2Ug4oCUCnRoZSAyMSBPY3QgQm9sdHogVGllciBDIGRvdWJsZSBjb3VudCBjYW1lIGZyb20gY29weWluZyBtb2RlbF9pLmNpZiBuZXh0IHRvIHRoZQpuZXN0ZWQgYm9sdHpfcmVzdWx0cy8gdHJlZSBhbmQgdGhlbiByZ2xvYi1pbmcgYm90aC4KIiIiCmltcG9ydCBzaHV0aWwsIHN5cywgcGF0aGxpYgp3b3JrID0gcGF0aGxpYi5QYXRoKHN5cy5hcmd2WzFdKTsgbW9kZWwgPSBzeXMuYXJndlsyXQpyYXcgPSB3b3JrLyJvdXRfcmF3Ii9tb2RlbAppZiBub3QgcmF3LmV4aXN0cygpOgogICAgcHJpbnQoIm5vIG91dF9yYXcgYXQiLCByYXcpOyBzeXMuZXhpdCgwKQpuID0gMApmb3Igam9iIGluIHNvcnRlZChwIGZvciBwIGluIHJhdy5pdGVyZGlyKCkgaWYgcC5pc19kaXIoKSk6CiAgICBkZXN0ID0gd29yay8ib3V0Ii9tb2RlbC9qb2IubmFtZQogICAgY2lmcyA9IHNvcnRlZChqb2Iucmdsb2IoIiouY2lmIikpCiAgICBpZiBub3QgY2lmczoKICAgICAgICBjb250aW51ZQogICAgZGVzdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBmb3IgaSwgYyBpbiBlbnVtZXJhdGUoY2lmcyk6CiAgICAgICAgc2h1dGlsLmNvcHkoYywgZGVzdC9mInNhbXBsZV97aX0uY2lmIik7IG4gKz0gMQogICAgcHJpbnQoZiIgIHtqb2IubmFtZX06IHtsZW4oY2lmcyl9IikKcHJpbnQoZiJjb2xsZWN0ZWQge259IHN0cnVjdHVyZXMgaW50byB7d29ya30vb3V0L3ttb2RlbH0iKQo=",
"mk_control_px.py": "aW1wb3J0IGpzb24sIHBhdGhsaWIsIHN5cwpXID0gcGF0aGxpYi5QYXRoKHN5cy5hcmd2WzFdIGlmIGxlbihzeXMuYXJndikgPiAxIGVsc2UgIndvcmtfY29udHJvbCIpCklPTlMgPSB7Ik1HIn0KbWFuID0ganNvbi5sb2FkKG9wZW4oVy8ibWFuaWZlc3QuanNvbiIpKQpvdXQgPSBXLyJpbnB1dHMiLyJwcm90ZW5peCI7IG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCk9ORSA9IHsiQUxBIjoiQSIsIkFSRyI6IlIiLCJBU04iOiJOIiwiQVNQIjoiRCIsIkNZUyI6IkMiLCJHTE4iOiJRIiwiR0xVIjoiRSIsIkdMWSI6IkciLAogICAgICAgIkhJUyI6IkgiLCJJTEUiOiJJIiwiTEVVIjoiTCIsIkxZUyI6IksiLCJNRVQiOiJNIiwiUEhFIjoiRiIsIlBSTyI6IlAiLCJTRVIiOiJTIiwKICAgICAgICJUSFIiOiJUIiwiVFJQIjoiVyIsIlRZUiI6IlkiLCJWQUwiOiJWIn0Kc3lzLnBhdGguaW5zZXJ0KDAsICIvY29udGVudCIpCmZyb20ga3Jhc19ob2xkb3V0IGltcG9ydCBlbnRpdHlfc2VxCm1hZGUsIGJhZCA9IDAsIFtdCmZvciBtIGluIG1hbjoKICAgIGUgPSBtWyJlbnRyeSJdCiAgICBzZXEsIF8gPSBlbnRpdHlfc2VxKFcvInN0cnVjdHVyZXMiL2Yie2V9LmNpZiIsIG1bImNoYWluIl0pCiAgICBwb3MgPSBtWyJzZXFfaW5kZXgiXQogICAgaWYgc2VxW3Bvcy0xXSAhPSBPTkUuZ2V0KG1bInJlcyJdLCAiWCIpOgogICAgICAgIGJhZC5hcHBlbmQoZiJ7ZX06IHNlcVt7cG9zfV09e3NlcVtwb3MtMV19ICE9IHttWydyZXMnXX0iKTsgY29udGludWUKICAgIHNlcXMgPSBbeyJwcm90ZWluQ2hhaW4iOiB7InNlcXVlbmNlIjogc2VxLCAiY291bnQiOiAxfX0sCiAgICAgICAgICAgIHsibGlnYW5kIjogeyJsaWdhbmQiOiBtWyJzbWlsZXMiXSwgImNvdW50IjogMX19XQogICAgZm9yIGMgaW4gbVsiY29mYWN0b3JzIl06CiAgICAgICAgc2Vxcy5hcHBlbmQoeyJpb24iOiB7ImlvbiI6IGMsICJjb3VudCI6IDF9fSBpZiBjIGluIElPTlMKICAgICAgICAgICAgICAgICAgICBlbHNlIHsibGlnYW5kIjogeyJsaWdhbmQiOiBmIkNDRF97Y30iLCAiY291bnQiOiAxfX0pCiAgICBqb2IgPSB7Im5hbWUiOiBmIntlfV9zbWlsZXMiLCAic2VxdWVuY2VzIjogc2Vxc30KICAgIChvdXQvZiJ7ZX1fc21pbGVzLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoW2pvYl0sIGluZGVudD0yKSk7IG1hZGUgKz0gMQpwcmludChmIndyb3RlIHttYWRlfSBjb250cm9sIHByb3Rlbml4IGpvYiBmaWxlcyAoc21pbGVzIGFybSBvbmx5KSIpCmZvciBiIGluIGJhZDogcHJpbnQoIiAgU0tJUCIsIGIpCmFzc2VydCBub3QgYmFkLCAibWFuaWZlc3Qvc2VxdWVuY2UgbWlzbWF0Y2giCg==",
"check_px_bonds.py": "aW1wb3J0IGpzb24sIHN5cwpPTkUgPSB7IkFMQSI6IkEiLCJBUkciOiJSIiwiQVNOIjoiTiIsIkFTUCI6IkQiLCJDWVMiOiJDIiwiR0xOIjoiUSIsIkdMVSI6IkUiLCJHTFkiOiJHIiwKICAgICAgICJISVMiOiJIIiwiSUxFIjoiSSIsIkxFVSI6IkwiLCJMWVMiOiJLIiwiTUVUIjoiTSIsIlBIRSI6IkYiLCJQUk8iOiJQIiwiU0VSIjoiUyIsCiAgICAgICAiVEhSIjoiVCIsIlRSUCI6IlciLCJUWVIiOiJZIiwiVkFMIjoiViJ9ClcgPSBzeXMuYXJndlsxXQptYW4gPSB7bVsiZW50cnkiXTogbSBmb3IgbSBpbiBqc29uLmxvYWQob3BlbihmIntXfS9tYW5pZmVzdC5qc29uIikpfQphbGxvayA9IFRydWUKZm9yIGUsIG0gaW4gbWFuLml0ZW1zKCk6CiAgICBqID0ganNvbi5sb2FkKG9wZW4oZiJ7V30vaW5wdXRzL3Byb3Rlbml4L3tlfV9ib25kLmpzb24iKSlbMF0KICAgIHNlcSA9IGpbInNlcXVlbmNlcyJdWzBdWyJwcm90ZWluQ2hhaW4iXVsic2VxdWVuY2UiXQogICAgYmQgPSBqWyJjb3ZhbGVudF9ib25kcyJdWzBdCiAgICBwb3MsIGV4cCA9IGludChiZFsicG9zaXRpb24xIl0pLCBPTkVbbVsicmVzIl1dCiAgICBvayA9IHNlcVtwb3MtMV0gPT0gZXhwIGFuZCBiZFsiYXRvbTIiXSA9PSBtWyJsYXRvbV9jaGFpIl0gYW5kIGJkWyJlbnRpdHkyIl0gPT0gIjIiCiAgICBhbGxvayAmPSBvawogICAgcHJpbnQoZiJ7ZTo8N30gYXV0aCB7bVsncmVzbnVtJ106PjR9IHBvc24ge3Bvczo+NH0gZXhwZWN0IHtleHB9IGdvdCB7c2VxW3Bvcy0xXX0geydPSycgaWYgb2sgZWxzZSAnRkFJTCd9ICB7YmRbJ2F0b20yJ119IikKYXNzZXJ0IGFsbG9rLCAicG9zaXRpb24gb3IgYXRvbSBtaXNtYXRjaCAtIHN0b3AiCnByaW50KCJhbGwgYm9uZCBkZWNsYXJhdGlvbnMgY2hlY2sgb3V0IikK",
"run_manifest.csv": "cGRiX2lkLHRpZXIscmVzaWR1ZV9uYW1lLHJlc2lkdWVfbnVtLHJlc2lkdWVfYXRvbSxsaWdhbmRfY29tcF9pZCxsaWdhbmRfYXRvbSxkZXBvc2l0ZWRfYm9uZF9BLHJlbGVhc2VfZGF0ZSxwb3N0X2JvbHR6Mixwb3N0X3Byb3Rlbml4XzIwMjUwNjMwCjlCQUksQV9ub25DeXNfVHlyLFRZUiw3MSxPSCxXTVUsUywxLjY2OSwyMDI1LTA0LTA5LHRydWUsZmFsc2UKOUJBSixBX25vbkN5c19UeXIsVFlSLDcxLE9ILEExQUs4LFMwMiwxLjY2MSwyMDI1LTA0LTA5LHRydWUsZmFsc2UKOUJBSyxBX25vbkN5c19UeXIsVFlSLDcxLE9ILFdOMyxTLDEuNjQ1LDIwMjUtMDQtMDksdHJ1ZSxmYWxzZQo4VDRWLEFfbm9uQ3lzX0FzcCxBU1AsMTIsT0QyLFk2MyxDMjQsMS40NjYsMjAyNC0wMy0xMyx0cnVlLGZhbHNlCjlHQkosQV9ub25DeXNfQXNwLEFTUCwxMixDRyxBMUlKNyxONDEsMS42MzIsMjAyNS0wNC0zMCx0cnVlLGZhbHNlCjlRUTAsQV9ub25DeXNfQXNwLEFTUCwxMixPRDIsQTFJOUUsQzQ2LDEuNDM0LDIwMjUtMDctMjMsdHJ1ZSx0cnVlCjlITVIsQl9DeXNfcG9zdFByb3Rlbml4LENZUyw5MixTRyxBMUlXRSxDMTksMS43NzEsMjAyNS0xMi0yNCx0cnVlLHRydWUKOUk3WSxCX0N5c19wb3N0UHJvdGVuaXgsQ1lTLDEzLFNHLEExSTA4LEMyMSwxLjgxNCwyMDI1LTExLTEyLHRydWUsdHJ1ZQo5TjQ0LEJfQ3lzX3Bvc3RQcm90ZW5peCxDWVMsMTIsU0csQTFCVjcsQzcsMS43OTksMjAyNS0xMi0xMCx0cnVlLHRydWUKOU5aTSxCX0N5c19wb3N0UHJvdGVuaXgsQ1lTLDEyLFNHLEExQjdQLEMzNywxLjgzMCwyMDI1LTExLTA1LHRydWUsdHJ1ZQo5TlpOLEJfQ3lzX3Bvc3RQcm90ZW5peCxDWVMsMTIsU0csQTFCN1EsQzM2LDEuODI4LDIwMjUtMTEtMDUsdHJ1ZSx0cnVlCjlQWkYsQl9DeXNfcG9zdFByb3Rlbml4LENZUywxMixTRyxBMUNRMixDMSwxLjk0MywyMDI2LTAzLTExLHRydWUsdHJ1ZQo5UFpZLEJfQ3lzX3Bvc3RQcm90ZW5peCxDWVMsMTIsU0csQTFBV1IsQzE3LDEuODQ1LDIwMjYtMDMtMTEsdHJ1ZSx0cnVlCjhTOEMsQ19DeXNfcG9zdEJvbHR6MixDWVMsMTIsU0csQTFINVUsQzQzLDEuODI1LDIwMjQtMDctMTAsdHJ1ZSxmYWxzZQo4VjM5LENfQ3lzX3Bvc3RCb2x0ejIsQ1lTLDEyLFNHLFk4TixDMzEsMS44MTQsMjAyNC0xMi0xOCx0cnVlLGZhbHNlCjhWM0EsQ19DeXNfcG9zdEJvbHR6MixDWVMsMTIsU0csWThOLEMzMSwxLjgxNCwyMDI0LTEyLTE4LHRydWUsZmFsc2UKOURNTSxDX0N5c19wb3N0Qm9sdHoyLENZUywxMixTRyxBMUFXUixDMTcsMS44MjksMjAyNS0wNS0yMSx0cnVlLGZhbHNlCjlFOUgsQ19DeXNfcG9zdEJvbHR6MixDWVMsMTIsU0csQTFCSDYsQzE2LDEuODA5LDIwMjUtMDItMjYsdHJ1ZSxmYWxzZQo5RTlJLENfQ3lzX3Bvc3RCb2x0ejIsQ1lTLDEyLFNHLEExQkg1LEMzMCwxLjgwOCwyMDI1LTAyLTI2LHRydWUsZmFsc2UKOUtQTSxDX0N5c19wb3N0Qm9sdHoyLENZUywxMixTRyxBMUw2QyxDMzAsMS43NjksMjAyNS0wMy0xMix0cnVlLGZhbHNlCjlLUE4sQ19DeXNfcG9zdEJvbHR6MixDWVMsMTIsU0csQTFMNkIsQzMwLDEuNzc1LDIwMjUtMDMtMTIsdHJ1ZSxmYWxzZQo=",
"control_manifest.csv": "cGRiX2lkLHRpZXIscmVzaWR1ZV9uYW1lLHJlc2lkdWVfbnVtLHJlc2lkdWVfYXRvbSxsaWdhbmRfY29tcF9pZCxsaWdhbmRfYXRvbSxkZXBvc2l0ZWRfYm9uZF9BLHJlbGVhc2VfZGF0ZSxwb3N0X2JvbHR6Mixwb3N0X3Byb3Rlbml4XzIwMjUwNjMwDQo0TFVDLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRywyMEcsUzE5LDIuMDM1LDIwMTMtMTEtMjcsZmFsc2UsZmFsc2UNCjRMVjYsRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLDIwSCxTMTcsMi4wNDQsMjAxMy0xMS0yNyxmYWxzZSxmYWxzZQ0KNExZSCxEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csMjFGLEMxNywxLjgyNiwyMDEzLTExLTI3LGZhbHNlLGZhbHNlDQo0TFlKLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRywyMUYsQzE3LDEuODMxLDIwMTMtMTEtMjcsZmFsc2UsZmFsc2UNCjRNMVcsRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLDIxUixDMjUsMS44MjUsMjAxMy0xMS0yNyxmYWxzZSxmYWxzZQ0KNE0yMixEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csMjJDLEMxOCwxLjgzMiwyMDEzLTExLTI3LGZhbHNlLGZhbHNlDQo0Tk1NLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyxZOVosQzEzLDEuODA0LDIwMTQtMDYtMDQsZmFsc2UsZmFsc2UNCjRQWlksRF9DeXNfcHJlQ2hhaTEsQ1lTLDcwLFNHLDJYUixDQUgsMS43NzAsMjAxNC0wOS0xMCxmYWxzZSxmYWxzZQ0KNFBaWixEX0N5c19wcmVDaGFpMSxDWVMsMzksU0csMlhPLFNBSCwyLjA3MSwyMDE0LTA5LTEwLGZhbHNlLGZhbHNlDQo0UTAxLERfQ3lzX3ByZUNoYWkxLENZUywzOSxTRywyWEgsU0FILDIuMDI1LDIwMTQtMDktMTAsZmFsc2UsZmFsc2UNCjRRMDIsRF9DeXNfcHJlQ2hhaTEsQ1lTLDM5LFNHLDJYRyxTQUcsMi4wNjEsMjAxNC0wOS0xMCxmYWxzZSxmYWxzZQ0KNUYyRSxEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csNVVULEMxNSwxLjc1NSwyMDE2LTAxLTEzLGZhbHNlLGZhbHNlDQo1S1lLLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyw2WkQsQzMyLDEuODA1LDIwMTctMDQtMTIsZmFsc2UsZmFsc2UNCjVWNlMsRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLDhZRCxDMjcsMS44MjgsMjAxNy0wNi0yOCxmYWxzZSxmYWxzZQ0KNVY2VixEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csOFlBLEMyNiwxLjc1OCwyMDE3LTA2LTI4LGZhbHNlLGZhbHNlDQo1VjcxLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyw4WkcsQzI4LDEuODI4LDIwMTctMDgtMjMsZmFsc2UsZmFsc2UNCjVWOUwsRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLDkxRCxDMTksMS42MDIsMjAxNy0wOC0yMyxmYWxzZSxmYWxzZQ0KNVY5TyxEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csOTFHLEMyMCwxLjQxNCwyMDE3LTA4LTIzLGZhbHNlLGZhbHNlDQo1VjlVLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyw5MVMsQzIwLDEuNzk2LDIwMTgtMDItMDcsZmFsc2UsZmFsc2UNCjVWQk0sRF9DeXNfcHJlQ2hhaTEsQ1lTLDcyLFNHLDkyVixTMjQsMi4wNjAsMjAxNy0xMC0yNSxmYWxzZSxmYWxzZQ0KNVlYWixEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csOTRDLEMxNCwxLjYxNCwyMDE4LTA0LTE4LGZhbHNlLGZhbHNlDQo1WVkxLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyw5NEYsQzE0LDEuNjE3LDIwMTgtMDQtMTgsZmFsc2UsZmFsc2UNCjZBUkssRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLEJRRCxDMDcsMS43NzcsMjAxOC0wMS0zMSxmYWxzZSxmYWxzZQ0KNkIwVixEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csQzhHLEMxMiwxLjY3OCwyMDE4LTA1LTE2LGZhbHNlLGZhbHNlDQo2QjBZLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyw4WkcsQzI4LDEuNjcxLDIwMTgtMDUtMTYsZmFsc2UsZmFsc2UNCjZOMkosRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLEs5TSxDMzAsMS43MTgsMjAxOC0xMi0xMixmYWxzZSxmYWxzZQ0KNk4ySyxEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csSzlKLEMyNiwxLjcyNywyMDE4LTEyLTEyLGZhbHNlLGZhbHNlDQo2T0lNLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyxNT1YsQzI1LDEuODA1LDIwMTktMTEtMDYsZmFsc2UsZmFsc2UNCjZQOFcsRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLE82NyxDMDEsMS44MTAsMjAxOS0wOC0yOCxmYWxzZSxmYWxzZQ0KNlA4WCxEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csTzVWLEMwMSwxLjgxMiwyMDE5LTA4LTI4LGZhbHNlLGZhbHNlDQo2UDhZLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyxPNVksQzAxLDEuODEwLDIwMTktMDgtMjgsZmFsc2UsZmFsc2UNCjZQOFosRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLE81UyxDMDEsMS44MTIsMjAxOS0wOC0yOCxmYWxzZSxmYWxzZQ0KNlBHTyxEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csT0oxLEMwMSwxLjgwNCwyMDE5LTEyLTI1LGZhbHNlLGZhbHNlDQo2UEdQLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyxPSFksQzI3LDEuODA4LDIwMTktMTItMjUsZmFsc2UsZmFsc2UNCjZUNUIsRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLE83SyxDLDEuODAzLDIwMjAtMDItMjYsZmFsc2UsZmFsc2UNCjZUQU0sRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLE1aUSxDMTYsMS43NTUsMjAyMC0wNC0wOCxmYWxzZSxmYWxzZQ0KNlRBTixEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csTVpOLEMxNywxLjcyNiwyMDIwLTA0LTA4LGZhbHNlLGZhbHNlDQo2VVNYLERfQ3lzX3ByZUNoYWkxLENZUywxMixTRyxNMVIsQzMyLDEuNjUxLDIwMjAtMDQtMjIsZmFsc2UsZmFsc2UNCjZVU1osRF9DeXNfcHJlQ2hhaTEsQ1lTLDEyLFNHLFFINCxDMzksMS42ODUsMjAyMC0wNC0yMixmYWxzZSxmYWxzZQ0KNlVUMCxEX0N5c19wcmVDaGFpMSxDWVMsMTIsU0csTTFYLEMzOSwxLjY0NywyMDIwLTA0LTIyLGZhbHNlLGZhbHNlDQo=",
"queue.sh": "IyEvYmluL2Jhc2gKIyBSZS1ydW4gb2YgdGhlIGFybXMgd2hvc2UgcGVyLXNhbXBsZSBkYXRhIHdhcyBsb3N0IG9uIDIyIE9jdCAyMDI2LgojIE9yZGVyIGlzIGJ5IGhvdyBtdWNoIG9mIHRoZSBwYXBlciBkZXBlbmRzIG9uIGVhY2ggYXJtLgpjZCAvY29udGVudApleHBvcnQgTEFZRVJOT1JNX1RZUEU9dG9yY2gKZXhwb3J0IERFU1Q9JHtERVNUOi0vY29udGVudC9kcml2ZS9NeURyaXZlL2tyYXNfY29mb2xkX3JlcnVuXzIwMjYtMTAtMjN9ClBYPS9jb250ZW50L3B4L2Jpbi9wcm90ZW5peApCWj0vY29udGVudC9iei9iaW4vYm9sdHoKbG9nKCl7IGVjaG8gIlskKGRhdGUgLXUgKyVIOiVNOiVTKVpdICQqIiB8IHRlZSAtYSAvY29udGVudC9xdWV1ZS5sb2c7IH0KUFhGTEFHUz0iLXMgNDIgLWUgNSAtcCAyMDAgLWMgMTAgLS11c2VfZGVmYXVsdF9wYXJhbXMgdHJ1ZSIKQlpGTEFHUz0iLS1kaWZmdXNpb25fc2FtcGxlcyA1IC0tc2VlZCA0MiAtLW5vX2tlcm5lbHMgLS1vdXRwdXRfZm9ybWF0IG1tY2lmIC0tdXNlX21zYV9zZXJ2ZXIiCgpydW5fcHggKCkgeyAjIHJ1bl9weCBNT0RFTCBJTlBVVF9KU09OIE9VVF9SQVdfRElSCiAgbG9jYWwgbz0kMwogIFsgIiQoZmluZCAkbyAtbmFtZSAnKi5jaWYnIDI+L2Rldi9udWxsIHwgd2MgLWwpIiAtZ3QgMCBdICYmIHsgbG9nICJza2lwICQoYmFzZW5hbWUgJG8pIjsgcmV0dXJuOyB9CiAgbWtkaXIgLXAgJG87IGxvZyAicHg6JDEgJChiYXNlbmFtZSAkbykiCiAgJFBYIHByZWQgLWkgJDIgLW8gJG8gLW4gJDEgJFBYRkxBR1MgPj4gL2NvbnRlbnQvc19weC5sb2cgMj4mMQogIGxvZyAiICAgLT4gJChmaW5kICRvIC1uYW1lICcqLmNpZicgfCB3YyAtbCkgY2lmcyIKfQpydW5fYnogKCkgeyAjIHJ1bl9ieiBZQU1MIE9VVF9SQVdfRElSCiAgbG9jYWwgbz0kMgogIFsgIiQoZmluZCAkbyAtbmFtZSAnKi5jaWYnIDI+L2Rldi9udWxsIHwgd2MgLWwpIiAtZ3QgMCBdICYmIHsgbG9nICJza2lwICQoYmFzZW5hbWUgJG8pIjsgcmV0dXJuOyB9CiAgbWtkaXIgLXAgJG87IGxvZyAiYnogJChiYXNlbmFtZSAkbykiCiAgJEJaIHByZWRpY3QgJDEgLS1vdXRfZGlyICRvICRCWkZMQUdTID4+IC9jb250ZW50L3NfYnoubG9nIDI+JjEKICBsb2NhbCBuPSQoZmluZCAkbyAtbmFtZSAnKi5jaWYnIHwgd2MgLWwpCiAgaWYgWyAiJG4iIC1lcSAwIF07IHRoZW4gbG9nICIgICAtPiBGQUlMRUQ6ICQoZ3JlcCAtRSAnRXJyb3J8ZXJyb3J8bm90IGZvdW5kJyAvY29udGVudC9zX2J6LmxvZyB8IHRhaWwgLTEgfCBjdXQgLWMxLTE2MCkiOyBlbHNlIGxvZyAiICAgLT4gJG4gY2lmcyI7IGZpCn0KCmxvZyAiPT09IFBSRVBBUkUgPT09IgpweXRob24ga3Jhc19ob2xkb3V0LnB5IHByZXBhcmUgd29ya19weCAtLW1hbmlmZXN0IHJ1bl9tYW5pZmVzdC5jc3YgMj4mMSB8IHRlZSAtYSAvY29udGVudC9xdWV1ZS5sb2cgfCB0YWlsIC0zCnB5dGhvbiBrcmFzX2hvbGRvdXQucHkgcHJlcGFyZSB3b3JrX2NvbnRyb2wgLS1tYW5pZmVzdCBjb250cm9sX21hbmlmZXN0LmNzdiAtLXRpZXIgRCAyPiYxIHwgdGVlIC1hIC9jb250ZW50L3F1ZXVlLmxvZyB8IHRhaWwgLTMKcHl0aG9uIHByb3Rlbml4X2FybS5weSBwcmVwYXJlIHdvcmtfcHggMj4mMSB8IHRhaWwgLTIgfCB0ZWUgLWEgL2NvbnRlbnQvcXVldWUubG9nCnB5dGhvbiBta19jb250cm9sX3B4LnB5IHdvcmtfY29udHJvbCAyPiYxIHwgdGFpbCAtMiB8IHRlZSAtYSAvY29udGVudC9xdWV1ZS5sb2cKcHl0aG9uIGNoZWNrX3B4X2JvbmRzLnB5IHdvcmtfcHggMj4mMSB8IHRlZSAtYSAvY29udGVudC9xdWV1ZS5sb2cgfCB0YWlsIC0zIHx8IHsgbG9nICJCT05EIENIRUNLIEZBSUxFRCAtIHN0b3BwaW5nIjsgZXhpdCAxOyB9CnRlc3QgJChscyB3b3JrX3B4L2lucHV0cy9wcm90ZW5peC8qLmpzb24gfCB3YyAtbCkgLWVxIDQyIHx8IHsgbG9nICJleHBlY3RlZCA0MiBwcm90ZW5peCBpbnB1dHMiOyBleGl0IDE7IH0KdGVzdCAkKGxzIHdvcmtfY29udHJvbC9pbnB1dHMvcHJvdGVuaXgvKi5qc29uIHwgd2MgLWwpIC1lcSA0MCB8fCB7IGxvZyAiZXhwZWN0ZWQgNDAgY29udHJvbCBpbnB1dHMiOyBleGl0IDE7IH0KbWtkaXIgLXAgd29ya19weF8yMDI1MDYzMC9zdHJ1Y3R1cmVzICYmIGNwIHdvcmtfcHgvbWFuaWZlc3QuanNvbiB3b3JrX3B4XzIwMjUwNjMwLyAmJiBjcCB3b3JrX3B4L3N0cnVjdHVyZXMvKiB3b3JrX3B4XzIwMjUwNjMwL3N0cnVjdHVyZXMvCmNwIC1yIHdvcmtfcHgvaW5wdXRzIHdvcmtfcHhfMjAyNTA2MzAvaW5wdXRzCm1rZGlyIC1wICIkREVTVCIgJiYgY3AgLWYgd29ya19weC9tYW5pZmVzdC5qc29uICIkREVTVC93b3JrX3B4X21hbmlmZXN0Lmpzb24iICYmIGNwIC1mIHdvcmtfY29udHJvbC9tYW5pZmVzdC5qc29uICIkREVTVC93b3JrX2NvbnRyb2xfbWFuaWZlc3QuanNvbiIKbG9nICJwcmVwYXJlIGRvbmUiCgpsb2cgIj09PSBTVEFHRSAxYTogcHJvdGVuaXgtMjAyNTA2MzAsIFRpZXIgQiArIEMsIGJvdGggYXJtcyAoMzAgam9icykgPT09Igpmb3IgZSBpbiA5SE1SIDlJN1kgOU40NCA5TlpNIDlOWk4gOVBaRiA5UFpZIDhTOEMgOFYzOSA4VjNBIDlETU0gOUU5SCA5RTlJIDlLUE0gOUtQTjsgZG8KICBmb3IgYSBpbiBzbWlsZXMgYm9uZDsgZG8gcnVuX3B4IHByb3Rlbml4X2Jhc2VfMjAyNTA2MzBfdjEuMC4wIHdvcmtfcHgvaW5wdXRzL3Byb3Rlbml4LyR7ZX1fJHthfS5qc29uIHdvcmtfcHhfMjAyNTA2MzAvb3V0X3Jhdy9wcm90ZW5peC8ke2V9XyR7YX07IGRvbmUKZG9uZQpweXRob24gY29sbGVjdC5weSB3b3JrX3B4XzIwMjUwNjMwIHByb3Rlbml4IHwgdGFpbCAtMSB8IHRlZSAtYSAvY29udGVudC9xdWV1ZS5sb2cKYmFzaCBwZXJzaXN0LnNoIFMxYSB3b3JrX3B4XzIwMjUwNjMwIHwgdGVlIC1hIC9jb250ZW50L3F1ZXVlLmxvZwpsb2cgIlNUQUdFIDFhIENPTVBMRVRFIgoKbG9nICI9PT0gU1RBR0UgMWI6IHByb3Rlbml4LTIwMjUwNjMwLCBUaWVyIEEsIGJvdGggYXJtcyAoMTIgam9icykgPT09Igpmb3IgZSBpbiA5QkFJIDlCQUogOUJBSyA4VDRWIDlHQkogOVFRMDsgZG8KICBmb3IgYSBpbiBzbWlsZXMgYm9uZDsgZG8gcnVuX3B4IHByb3Rlbml4X2Jhc2VfMjAyNTA2MzBfdjEuMC4wIHdvcmtfcHgvaW5wdXRzL3Byb3Rlbml4LyR7ZX1fJHthfS5qc29uIHdvcmtfcHhfMjAyNTA2MzAvb3V0X3Jhdy9wcm90ZW5peC8ke2V9XyR7YX07IGRvbmUKZG9uZQpweXRob24gY29sbGVjdC5weSB3b3JrX3B4XzIwMjUwNjMwIHByb3Rlbml4IHwgdGFpbCAtMSB8IHRlZSAtYSAvY29udGVudC9xdWV1ZS5sb2cKYmFzaCBwZXJzaXN0LnNoIFMxYiB3b3JrX3B4XzIwMjUwNjMwIHwgdGVlIC1hIC9jb250ZW50L3F1ZXVlLmxvZwpsb2cgIlNUQUdFIDFiIENPTVBMRVRFIgoKbG9nICI9PT0gU1RBR0UgMjogYm9sdHotMiwgYWxsIDIxIGVudHJpZXMsIGJvdGggYXJtcyAoNDIgam9iczsgQ0NELW1pc3NpbmcgYm9uZCBhcm1zIGZhaWwgYnkgZGVzaWduKSA9PT0iCmZvciBkIGluIHdvcmtfcHgvaW5wdXRzL2JvbHR6LyovOyBkbwogIG49JChiYXNlbmFtZSAkZCk7IHk9JChscyAkZCoueWFtbCAyPi9kZXYvbnVsbCB8IGhlYWQgLTEpOyBbIC16ICIkeSIgXSAmJiBjb250aW51ZQogIHJ1bl9ieiAkeSB3b3JrX3B4L291dF9yYXcvYm9sdHovJG4KZG9uZQpweXRob24gY29sbGVjdC5weSB3b3JrX3B4IGJvbHR6IHwgdGFpbCAtMSB8IHRlZSAtYSAvY29udGVudC9xdWV1ZS5sb2cKYmFzaCBwZXJzaXN0LnNoIFMyIHdvcmtfcHggfCB0ZWUgLWEgL2NvbnRlbnQvcXVldWUubG9nCmxvZyAiU1RBR0UgMiBDT01QTEVURSIKCmxvZyAiPT09IFNUQUdFIDM6IGJvbHR6LTIsIDQwIGNvbnRyb2xzLCBzbWlsZXMgb25seSA9PT0iCmZvciBkIGluIHdvcmtfY29udHJvbC9pbnB1dHMvYm9sdHovKl9zbWlsZXMvOyBkbwogIG49JChiYXNlbmFtZSAkZCk7IHk9JChscyAkZCoueWFtbCAyPi9kZXYvbnVsbCB8IGhlYWQgLTEpOyBbIC16ICIkeSIgXSAmJiBjb250aW51ZQogIHJ1bl9ieiAkeSB3b3JrX2NvbnRyb2wvb3V0X3Jhdy9ib2x0ei8kbgpkb25lCnB5dGhvbiBjb2xsZWN0LnB5IHdvcmtfY29udHJvbCBib2x0eiB8IHRhaWwgLTEgfCB0ZWUgLWEgL2NvbnRlbnQvcXVldWUubG9nCmJhc2ggcGVyc2lzdC5zaCBTMyB3b3JrX2NvbnRyb2wgfCB0ZWUgLWEgL2NvbnRlbnQvcXVldWUubG9nCmxvZyAiU1RBR0UgMyBDT01QTEVURSIKCmxvZyAiPT09IFNUQUdFIDQ6IHByb3Rlbml4IHYxLjAuMCwgNDAgY29udHJvbHMsIHNtaWxlcyBvbmx5ID09PSIKZm9yIGYgaW4gd29ya19jb250cm9sL2lucHV0cy9wcm90ZW5peC8qX3NtaWxlcy5qc29uOyBkbwogIG49JChiYXNlbmFtZSAkZiAuanNvbik7IHJ1bl9weCBwcm90ZW5peF9iYXNlX2RlZmF1bHRfdjEuMC4wICRmIHdvcmtfY29udHJvbC9vdXRfcmF3L3Byb3Rlbml4LyRuCmRvbmUKcHl0aG9uIGNvbGxlY3QucHkgd29ya19jb250cm9sIHByb3Rlbml4IHwgdGFpbCAtMSB8IHRlZSAtYSAvY29udGVudC9xdWV1ZS5sb2cKYmFzaCBwZXJzaXN0LnNoIFM0IHdvcmtfY29udHJvbCB8IHRlZSAtYSAvY29udGVudC9xdWV1ZS5sb2cKbG9nICJTVEFHRSA0IENPTVBMRVRFIgpsb2cgIkFMTCBTVEFHRVMgQ09NUExFVEUiCg==",
"persist.sh": "IyEvYmluL2Jhc2gKIyBwZXJzaXN0LnNoIFNUQUdFIFdPUktESVIgIOKAlCBzY29yZSwgcG9zZS1STVNELCB0aGVuIGNvcHkgZXZlcnl0aGluZyB0aGF0IG1hdHRlcnMgdG8gRHJpdmUuClNUQUdFPSQxOyBXT1JLPSQyCmNkIC9jb250ZW50CkRFU1Q9JHtERVNUOi0vY29udGVudC9kcml2ZS9NeURyaXZlL2tyYXNfY29mb2xkX3JlcnVuXzIwMjYtMTAtMjN9Cm1rZGlyIC1wICIkREVTVC8kV09SSyIKcHl0aG9uIC9jb250ZW50L2tyYXNfaG9sZG91dC5weSBzY29yZSAkV09SSyA+ICRXT1JLL3Njb3JlXyR7U1RBR0V9LnR4dCAyPiYxIHx8IHRydWUKcHl0aG9uIC9jb250ZW50L3Bvc2VybXMucHkgJFdPUksgPiAkV09SSy9wb3Nlcm1zXyR7U1RBR0V9LnR4dCAyPiYxIHx8IHRydWUKY3AgLWYgJFdPUksvcmVzdWx0cy5jc3YgICAiJERFU1QvJFdPUksvcmVzdWx0cy5jc3YiICAgMj4vZGV2L251bGwKY3AgLWYgJFdPUksvcG9zZV9ybXNkLmNzdiAiJERFU1QvJFdPUksvcG9zZV9ybXNkLmNzdiIgMj4vZGV2L251bGwKY3AgLWYgJFdPUksvbWFuaWZlc3QuanNvbiAiJERFU1QvJFdPUksvbWFuaWZlc3QuanNvbiIgMj4vZGV2L251bGwKY3AgLWYgJFdPUksvc2NvcmVfJHtTVEFHRX0udHh0ICRXT1JLL3Bvc2VybXNfJHtTVEFHRX0udHh0ICIkREVTVC8kV09SSy8iIDI+L2Rldi9udWxsCnRhciAtY3pmICIkREVTVC8ke1dPUkt9X291dF8ke1NUQUdFfS50YXIuZ3oiIC1DIC9jb250ZW50ICRXT1JLL291dCAkV09SSy9pbnB1dHMgMj4vZGV2L251bGwKY3AgLWYgL2NvbnRlbnQvcXVldWUubG9nIC9jb250ZW50L3MqLmxvZyAiJERFU1QvIiAyPi9kZXYvbnVsbAplY2hvICIkKGRhdGUgLXUgKyVGVCVUWikgJFNUQUdFICRXT1JLIHJvd3M9JCgoICQod2MgLWwgPCAkV09SSy9yZXN1bHRzLmNzdiAyPi9kZXYvbnVsbCB8fCBlY2hvIDEpIC0gMSApKSIgPj4gIiRERVNUL1NUQUdFU19DT01QTEVURS50eHQiCnN5bmMKZWNobyAicGVyc2lzdGVkICRTVEFHRSAtPiAkREVTVCIK",
"install.sh": "IyEvYmluL2Jhc2gKc2V0IC1lCmNkIC9jb250ZW50CnBpcCBpbnN0YWxsIC1xIGdlbW1pIHJka2l0IHJlcXVlc3RzIG51bXB5IDI+JjEgfCB0YWlsIC0xCmVjaG8gIi0tLSBwcm90ZW5peCB2ZW52IgpweXRob24zIC1tIHZlbnYgLS1zeXN0ZW0tc2l0ZS1wYWNrYWdlcyAvY29udGVudC9weAovY29udGVudC9weC9iaW4vcGlwIGluc3RhbGwgLXEgLS11cGdyYWRlIHBpcCAyPiYxIHwgdGFpbCAtMQovY29udGVudC9weC9iaW4vcGlwIGluc3RhbGwgLXEgLS11cGdyYWRlIHByb3Rlbml4IC0taW5kZXgtdXJsIGh0dHBzOi8vcHlwaS5vcmcvc2ltcGxlIDI+JjEgfCB0YWlsIC0yCmVjaG8gIi0tLSBib2x0eiB2ZW52IgpweXRob24zIC1tIHZlbnYgLS1zeXN0ZW0tc2l0ZS1wYWNrYWdlcyAvY29udGVudC9iegovY29udGVudC9iei9iaW4vcGlwIGluc3RhbGwgLXEgLS11cGdyYWRlIHBpcCAyPiYxIHwgdGFpbCAtMQovY29udGVudC9iei9iaW4vcGlwIGluc3RhbGwgLXEgLVUgYm9sdHogMj4mMSB8IHRhaWwgLTIKZWNobyAiLS0tIGNoZWNrIgpMQVlFUk5PUk1fVFlQRT10b3JjaCAvY29udGVudC9weC9iaW4vcHJvdGVuaXggLS1oZWxwIDI+JjEgfCBoZWFkIC0zCi9jb250ZW50L2J6L2Jpbi9ib2x0eiBwcmVkaWN0IC0taGVscCAyPiYxIHwgaGVhZCAtMgplY2hvIElOU1RBTExfT0sK"
}
for name, b64 in BLOBS.items():
    pathlib.Path(name).write_bytes(base64.b64decode(b64))
    print(f"wrote {name} {pathlib.Path(name).stat().st_size} bytes")
import hashlib; print("kras_holdout.py sha256:", hashlib.sha256(open('kras_holdout.py','rb').read()).hexdigest()[:16])

## 3 · Install (two venvs, as on 22 Oct; ~10 min)

In [ ]:
!bash /content/install.sh

## 4 · Launch the queue (prepare → S1a → S1b → S2 → S3 → S4)

In [ ]:
import subprocess, os
os.chdir('/content')
os.environ['DEST'] = '/content/drive/MyDrive/kras_cofold_rerun_2026-10-23'
subprocess.run('pkill -f queue.sh; true', shell=True)
subprocess.Popen('nohup bash /content/queue.sh > /content/queue.out 2>&1 &', shell=True, env=os.environ)
print("queue started")

## 5 · Progress (re-run this cell)

In [ ]:
import subprocess, os
DEST='/content/drive/MyDrive/kras_cofold_rerun_2026-10-23'
print(subprocess.run('tail -n 25 /content/queue.log', shell=True, capture_output=True, text=True).stdout)
print('--- stages persisted:'); print(open(f'{DEST}/STAGES_COMPLETE.txt').read() if os.path.exists(f'{DEST}/STAGES_COMPLETE.txt') else '(none yet)')
print('--- Drive folder:', sorted(os.listdir(DEST)))

## 6 · After ALL STAGES COMPLETE: flush Drive and pull a local copy too

In [ ]:
from google.colab import drive, files
import shutil
shutil.make_archive('/content/kras_cofold_rerun_all', 'zip', '/content/drive/MyDrive/kras_cofold_rerun_2026-10-23')
drive.flush_and_unmount()
files.download('/content/kras_cofold_rerun_all.zip')